# Phase C — shared **Triplanar UNet** (M4)

Supersedes the 3-independent-net `phase_c3_planes_train.ipynb`. Same slice-based data
(the PL volumes from `phase_b_triplane_dataset.ipynb`, sliced into xy/yz/zx planes), but:

1. **One shared-weight UNet** predicts every orientation's planes — ~3x fewer params,
   learns orientation-agnostic propagation, better data efficiency. It carries an
   **orientation one-hot** (so it knows which stack it is predicting) and uses
   **GroupNorm** (stable across the three very different plane sizes) with per-orientation
   anisotropic pooling (the thin 17-voxel Y axis is never collapsed).
2. **Cross-plane communication** is a small **learned 3-D fusion head**: it takes the three
   orientation volume estimates and combines them at the shared 3-D frame, replacing the
   ad-hoc mean. This is the deterministic, physics-appropriate reading of the "Triplanar
   UNet" idea — no diffusion, no image/CLIP conditioning (there is nothing to sample and
   no input image; PL is one deterministic field given geometry+Tx+freq).

Objective is the proven 2-D one (masked-MSE + 0.1 gradient-L1); AMP, cosine, resume, early
stop, ONNX parity as before.

In [ ]:
#@title Run mode
RUN_MODE   = "full"   #@param ["full", "smoke"]
BASE       = 64       #@param {type:"integer"}
LR         = 1e-3     #@param {type:"number"}
BS         = 16       #@param {type:"integer"}
EPOCHS     = 120      #@param {type:"integer"}
PATIENCE   = 15       #@param {type:"integer"}
SLICES_PER_TX = 8     #@param {type:"integer"}
POOL_THRESH   = 24    #@param {type:"integer"}
FUSION     = True     #@param {type:"boolean"}   learned 3-D cross-plane fusion head
N_FUSE     = 48       #@param {type:"integer"}   train Tx used to fit the fusion head
N_TEST     = 24       #@param {type:"integer"}
NW         = 2        #@param {type:"integer"}
SEED       = 0
if RUN_MODE == "smoke":
    EPOCHS, SLICES_PER_TX, N_FUSE, N_TEST, NW = 2, 3, 4, 4, 0
print(f"RUN_MODE={RUN_MODE}  base={BASE}  epochs={EPOCHS}  fusion={FUSION}")

In [ ]:
#@title Setup
import os, sys, json, time, math, glob
from itertools import zip_longest
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

ROOT = "/content/drive/MyDrive/indoor-walk-test-main/Physics Engine/3D Map Physics/SIM V1 3D"  #@param {type:"string"}
try:
    from google.colab import drive; drive.mount("/content/drive")
except Exception:
    pass
ROOT = str(__import__("pathlib").Path(ROOT).expanduser().resolve())
sys.path.insert(0, ROOT)
import dataset_3d as D

torch.manual_seed(SEED); np.random.seed(SEED)
torch.backends.cudnn.benchmark = True
dev = "cuda" if torch.cuda.is_available() else "cpu"
man    = json.load(open(f"{ROOT}/manifest_3d.json"))
M      = np.load(f"{ROOT}/material_grid.npy")
inside = np.load(f"{ROOT}/inside_mask.npy")
norm   = D.load_norm(man)
CELL   = float(man["cell_size_m"])
PL_LO, PL_RNG = norm.pl_min_db, norm.pl_range_db
DATA   = f"{ROOT}/dataset"
CKPT_D = f"{ROOT}/checkpoints"; os.makedirs(CKPT_D, exist_ok=True)
WEB    = f"{ROOT}/web";         os.makedirs(WEB, exist_ok=True)

ORIENTS = list(D.PLANE_ORIENTS)                     # ("zx","xy","yz")
ORIENT_IDX = {o: i for i, o in enumerate(ORIENTS)}
CIN = len(D.INPUT_CHANNELS_PLANE) + len(ORIENTS)    # 10 + 3 = 13
POOLS = {o: (2 if D.plane_shape(M.shape, o)[0] >= POOL_THRESH else 1,
             2 if D.plane_shape(M.shape, o)[1] >= POOL_THRESH else 1) for o in ORIENTS}
print("device", dev, "| in_ch", CIN, "| pools", POOLS)

def triplane_input(tx, ff, orient, k):
    """(13,H,W): the 10-channel plane input + a 3-channel orientation one-hot."""
    x = D.plane_input(M, tx, ff, orient, k, CELL)
    oh = np.zeros((len(ORIENTS),) + x.shape[1:], np.float32)
    oh[ORIENT_IDX[orient]] = 1.0
    return np.concatenate([x, oh], 0)

## Dataset — one `TriPlaneDS` per orientation, one shared net

Same slicing as the 3-net version, plus the orientation one-hot. Three loaders feed the
single shared UNet; batches stay single-orientation (so shapes match), the net weights are
shared across all of them.

In [ ]:
#@title TriPlaneDS
sp = json.load(open(f"{DATA}/splits.json"))
assert sp.get("scene_sha", D.scene_sha(M)) == D.scene_sha(M), "splits.json is a different scene"

def _spread(valid, tx_fixed, k):
    if len(valid) <= k:
        return list(valid)
    pick = np.linspace(0, len(valid) - 1, k).round().astype(int)
    out = {int(valid[p]) for p in pick}
    out.add(int(valid[np.argmin(np.abs(np.asarray(valid) - tx_fixed))]))
    return sorted(out)

class TriPlaneDS(Dataset):
    def __init__(self, keep_pos, orient, *, full=False, slices_per_tx=SLICES_PER_TX):
        self.orient = orient; self.fa = D._ORIENT_FIXED_AXIS[orient]
        nfix = D.n_slices(M.shape, orient)
        self.valid = [k for k in range(nfix) if D.slice_plane(inside, orient, k).any()]
        keep = set(int(p) for p in keep_pos); self.pl, self.items = [], []
        for mp in D.list_shards(DATA):
            s = int(os.path.basename(mp).split("_")[1])
            pl, _t, meta = D.open_shard(DATA, s)
            si = len(self.pl); self.pl.append(pl)
            for i, pid in enumerate(meta["pos_id"]):
                if int(pid) not in keep:
                    continue
                tx = tuple(int(v) for v in meta["tx"][i]); ff = float(meta["freq_feat"][i])
                idxs = self.valid if full else _spread(self.valid, tx[self.fa], slices_per_tx)
                self.items += [(si, i, tx, ff, int(k)) for k in idxs]

    def __len__(self):
        return len(self.items)

    def __getitem__(self, j):
        si, i, tx, ff, k = self.items[j]
        y = D.slice_plane(np.asarray(self.pl[si][i], np.float32), self.orient, k)
        x = triplane_input(tx, ff, self.orient, k)
        m = D.slice_plane(inside, self.orient, k).astype(np.float32)
        return torch.from_numpy(x), torch.from_numpy(y[None]), torch.from_numpy(m[None])

def make_loaders(split, shuffle):
    dls = []
    for o in ORIENTS:
        ds = TriPlaneDS(sp[split], o)
        dls.append(DataLoader(ds, BS, shuffle=shuffle, num_workers=NW, pin_memory=(dev == "cuda")))
    return dls

tl = make_loaders("train", True); vl = make_loaders("val", False)
assert sum(len(d.dataset) for d in tl) > 0, "no training planes — run phase_b first"
for o, d in zip(ORIENTS, tl):
    print(f"{o}: train {len(d.dataset)} planes")

## Shared Triplanar UNet + fusion head + loss

In [ ]:
#@title Model + loss
def gnconv(ci, co, g=8):
    return nn.Sequential(nn.Conv2d(ci, co, 3, padding=1), nn.GroupNorm(min(g, co), co), nn.ReLU(True),
                         nn.Conv2d(co, co, 3, padding=1), nn.GroupNorm(min(g, co), co), nn.ReLU(True))

class TriplanarUNet(nn.Module):
    """One shared UNet for all orientations; `pool` is chosen per orientation at call time."""
    def __init__(self, cin, base=BASE):
        super().__init__()
        self.e1 = gnconv(cin, base); self.e2 = gnconv(base, base * 2); self.b = gnconv(base * 2, base * 4)
        self.d2 = gnconv(base * 4 + base * 2, base * 2); self.d1 = gnconv(base * 2 + base, base)
        self.out = nn.Conv2d(base, 1, 1)
    def _up(self, x, skip):
        x = F.interpolate(x, size=skip.shape[2:], mode="bilinear", align_corners=False)
        return torch.cat([x, skip], 1)
    def forward(self, x, pool):
        p = lambda t: F.max_pool2d(t, pool)
        e1 = self.e1(x); e2 = self.e2(p(e1)); b = self.b(p(e2))
        d2 = self.d2(self._up(b, e2)); d1 = self.d1(self._up(d2, e1))
        return torch.sigmoid(self.out(d1))

class FusionHead(nn.Module):
    """Cross-plane: combine the 3 orientation volume estimates at the shared 3-D frame."""
    def __init__(self, k=8):
        super().__init__()
        self.net = nn.Sequential(nn.Conv3d(3, k, 3, padding=1), nn.GroupNorm(min(4, k), k),
                                 nn.ReLU(True), nn.Conv3d(k, 1, 1))
    def forward(self, vols):                          # vols: (B,3,X,Y,Z) normalized
        return torch.sigmoid(self.net(vols))

def grad_l1(a, b):
    return (((a[:, :, 1:, :] - a[:, :, :-1, :]) - (b[:, :, 1:, :] - b[:, :, :-1, :])).abs().mean()
            + ((a[:, :, :, 1:] - a[:, :, :, :-1]) - (b[:, :, :, 1:] - b[:, :, :, :-1])).abs().mean())
def masked_mse(p, y, m):
    return (((p - y) ** 2) * m).sum() / m.sum().clamp(min=1)
def total_loss(p, y, m):
    return masked_mse(p, y, m) + 0.1 * grad_l1(p * m, y * m)

net = TriplanarUNet(CIN).to(dev)
print(f"shared UNet: {sum(p.numel() for p in net.parameters())/1e6:.1f} M params (one net for all 3 stacks)")

## Train the shared UNet

Batches from the three orientation loaders are interleaved into the same optimizer, so one
set of weights sees every orientation each epoch. Early stop on the overall val RMSE.

In [ ]:
#@title Train
amp_dtype = torch.bfloat16 if (dev == "cuda" and torch.cuda.is_bf16_supported()) else torch.float16

def lr_at(step, total, warm=0.1):
    t = min(step / max(total, 1), 1.0)
    if t < warm:
        return LR * max(t / warm, 1e-3)
    c = (t - warm) / (1 - warm)
    return LR * 0.02 + 0.5 * LR * 0.98 * (1 + math.cos(math.pi * c))

@torch.no_grad()
def val_rmse():
    net.eval(); out = {}
    for o, dl in zip(ORIENTS, vl):
        se = n = 0.0
        for x, y, m in dl:
            x, y, m = x.to(dev), y.to(dev), m.to(dev)
            with torch.autocast(dev, amp_dtype, enabled=(dev == "cuda")):
                p = net(x, POOLS[o]).float()
            se += float(((p - y) ** 2 * m).sum()); n += float(m.sum().clamp(min=1))
        out[o] = PL_RNG * math.sqrt(se / max(n, 1))
    out["all"] = float(np.mean([out[o] for o in ORIENTS]))
    return out

opt = torch.optim.AdamW(net.parameters(), LR, weight_decay=1e-4)
scaler = torch.amp.GradScaler(dev, enabled=(dev == "cuda" and amp_dtype == torch.float16))
CKPT, RES = f"{CKPT_D}/best_triplanar.pt", f"{CKPT_D}/resume_triplanar.pt"
total = EPOCHS * max(sum(len(d) for d in tl), 1)
start, best, bad, g = 0, 1e9, 0, 0
if os.path.exists(RES):
    st = torch.load(RES, map_location=dev)
    net.load_state_dict(st["net"]); opt.load_state_dict(st["opt"]); scaler.load_state_dict(st["scaler"])
    start, best, g = st["epoch"] + 1, st["best"], st["g"]; print(f"resumed at {start} (best {best:.2f})")

for ep in range(start, EPOCHS):
    net.train(); t0 = time.time()
    for batches in zip_longest(*[iter(d) for d in tl]):
        for o, b in zip(ORIENTS, batches):
            if b is None:
                continue
            x, y, m = (t.to(dev, non_blocking=True) for t in b)
            for pg in opt.param_groups:
                pg["lr"] = lr_at(g, total)
            with torch.autocast(dev, amp_dtype, enabled=(dev == "cuda")):
                loss = total_loss(net(x, POOLS[o]).float(), y, m)
            opt.zero_grad(set_to_none=True); scaler.scale(loss).backward()
            scaler.unscale_(opt); nn.utils.clip_grad_norm_(net.parameters(), 1.0)
            scaler.step(opt); scaler.update(); g += 1
    r = val_rmse()
    if r["all"] < best - 1e-4:
        best, bad = r["all"], 0; torch.save(net.state_dict(), CKPT)
    else:
        bad += 1
    torch.save(dict(net=net.state_dict(), opt=opt.state_dict(), scaler=scaler.state_dict(),
                    epoch=ep, best=best, g=g), RES)
    print(f"ep {ep+1:03d}/{EPOCHS}  val {r['all']:5.2f} dB  "
          f"[zx {r['zx']:.2f} xy {r['xy']:.2f} yz {r['yz']:.2f}]  best {best:5.2f}  ({time.time()-t0:.0f}s)")
    if bad >= PATIENCE:
        print("early stop"); break
net.load_state_dict(torch.load(CKPT, map_location=dev))
print(f"\nbest shared-UNet val {best:.2f} dB")

## Reconstruct + cross-plane fusion + volume RMSE

Reconstruct each orientation's volume by running the shared net over all its slices (one
batched forward per orientation), then either mean-compose or the learned 3-D fusion head.
The number that matters is volume RMSE vs the stored (engine) volumes — beat 14.47 dB.

In [ ]:
#@title Reconstruct, fuse, evaluate
@torch.no_grad()
def reconstruct(tx, ff):
    net.eval(); est = []
    for o in ORIENTS:
        n = D.n_slices(M.shape, o)
        xs = torch.from_numpy(np.stack([triplane_input(tx, ff, o, k) for k in range(n)])).to(dev)
        p = net(xs, POOLS[o])[:, 0].float().cpu().numpy()          # (n,H,W) normalized
        est.append(D.stack_planes([p[k] for k in range(n)], o))    # (X,Y,Z)
    return np.stack(est, 0).astype(np.float32)                     # (3,X,Y,Z)

def gather(split, n_max):
    out = []
    for mp in D.list_shards(DATA):
        s = int(os.path.basename(mp).split("_")[1])
        pl, _t, meta = D.open_shard(DATA, s)
        for i, pid in enumerate(meta["pos_id"]):
            if int(pid) in set(sp[split]):
                out.append((np.asarray(pl[i], np.float32), tuple(int(v) for v in meta["tx"][i]),
                            float(meta["freq_feat"][i]), float(meta["freq_mhz"][i])))
            if len(out) >= n_max:
                return out
    return out

fuse = None
if FUSION:
    tr_s = gather("train", N_FUSE)
    X = torch.from_numpy(np.stack([reconstruct(tx, ff) for _g, tx, ff, _f in tr_s]))
    Y = torch.from_numpy(np.stack([g for g, _t, _ff, _f in tr_s]))[:, None]
    Mk = torch.from_numpy(inside.astype(np.float32))[None, None]
    fuse = FusionHead().to(dev)
    fo = torch.optim.Adam(fuse.parameters(), 1e-3)
    for it in range(300):
        idx = torch.randint(0, len(X), (min(4, len(X)),))
        xb, yb = X[idx].to(dev), Y[idx].to(dev)
        loss = masked_mse(fuse(xb), yb, Mk.to(dev))
        fo.zero_grad(); loss.backward(); fo.step()
    print(f"fusion head trained (final masked-MSE {float(loss):.5f})")

def rmse_db(a, b, m):
    return float(np.sqrt((((a - b)[m]) * PL_RNG) ** 2 ).mean())

test = gather("test", N_TEST)
agg = {k: [] for k in ORIENTS + ["mean", "fused", "fspl"]}
for gt, tx, ff, fmhz in test:
    est = reconstruct(tx, ff)
    for i, o in enumerate(ORIENTS):
        agg[o].append(rmse_db(est[i], gt, inside))
    agg["mean"].append(rmse_db(est.mean(0), gt, inside))
    if fuse is not None:
        with torch.no_grad():
            fv = fuse(torch.from_numpy(est)[None].to(dev))[0, 0].cpu().numpy()
        agg["fused"].append(rmse_db(fv, gt, inside))
    d = D.distance_m(tx, D.voxel_coords(M.shape), CELL)
    agg["fspl"].append(rmse_db(np.clip((D.fspl_db(d, fmhz, man) - PL_LO) / PL_RNG, 0, 1), gt, inside))

print(f"\nvolume RMSE over {len(test)} test Tx (dB):")
for k in ORIENTS + ["mean"] + (["fused"] if fuse is not None else []) + ["fspl"]:
    if agg[k]:
        print(f"  {k:>6}: {np.mean(agg[k]):5.2f}")
print("full-3D surrogate baseline was 14.47 dB")

## Export ONNX + contracts

Three per-orientation graphs from the one shared net (pool baked per orientation) + the
fusion head, each with a `pl_unet2d_<orient>.json` contract.

In [ ]:
#@title Export
try:
    import onnxruntime as ort
except ModuleNotFoundError:
    import subprocess; subprocess.run([sys.executable, "-m", "pip", "install", "-q", "onnxruntime", "onnx"], check=True)
    import onnxruntime as ort

class OrientGraph(nn.Module):                      # bakes one orientation's pool for export
    def __init__(self, net, pool): super().__init__(); self.net, self.pool = net, pool
    def forward(self, x): return self.net(x, self.pool)

net.eval()
for o in ORIENTS:
    H, W = D.plane_shape(M.shape, o)
    p = f"{WEB}/pl_unet2d_{o}.onnx"
    g = OrientGraph(net, POOLS[o]).eval()
    torch.onnx.export(g, torch.zeros(1, CIN, H, W, device=dev), p, opset_version=17,
                      input_names=["x"], output_names=["y"], dynamic_axes={"x": {0: "n"}, "y": {0: "n"}})
    sess = ort.InferenceSession(p, providers=["CPUExecutionProvider"]); worst = 0.0
    for k, (x, _y, _m) in enumerate(vl[ORIENT_IDX[o]]):
        if k >= 3:
            break
        ref = g(x.to(dev)).float().cpu().numpy()
        got = sess.run(["y"], {"x": x.numpy().astype(np.float32)})[0]
        worst = max(worst, float(np.abs(ref - got).max() * PL_RNG))
    contract = D.surrogate_contract_plane(
        man, M, orient=o, bands=sp.get("train_bands_mhz", man["freqs_mhz"]),
        metrics=dict(volume_rmse_db=round(float(np.mean(agg[o])), 3)),
        extra=dict(shared_weights=True, orient_onehot=list(ORIENTS),
                   input_channels=list(D.INPUT_CHANNELS_PLANE) + [f"orient_{x}" for x in ORIENTS],
                   fusion="learned_3d" if fuse is not None else "mean", onnx_parity_db=round(worst, 4)))
    D.write_surrogate_contract(f"{WEB}/pl_unet2d_{o}.json", contract)
    print(f"[{o}] wrote onnx + json | parity {worst:.4f} dB {'OK' if worst <= 0.1 else 'FAIL'}")

if fuse is not None:
    fp = f"{WEB}/pl_fuse3d.onnx"
    torch.onnx.export(fuse.eval(), torch.zeros(1, 3, *M.shape, device=dev), fp, opset_version=17,
                      input_names=["v"], output_names=["y"])
    print(f"wrote {os.path.basename(fp)} (cross-plane fusion head)")
print("\ndone — one shared Triplanar UNet exported as 3 orientation graphs" + (" + fusion" if fuse is not None else ""))